# HaloRT - Getting Started

This notebook installs the `rendervous` (rdv) + `vulky` (vk) rendering
engine and this `halort` package, builds a small synthetic cloud volume
and HDR-style environment (no external dataset needed), and renders it
with all three supported models: **HaloRT**, **Denoiser**, and
**DenoiserX**.

> A reference to a Hugging Face dataset with real cloud volumes will be
> added to the main README later - swap the synthetic cloud in this
> notebook for one loaded from there once available.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q rendervous
    import os
    if not os.path.exists("HaloRT"):
        !git clone --depth 1 https://github.com/rendervous/HaloRT.git
    %cd HaloRT
    import sys
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    print("Installed rendervous and cloned HaloRT (brings pretrained/ along) into", os.getcwd())
else:
    print("Not running in Colab - make sure rendervous is installed and you're running from the repo root (so pretrained/ resolves)")


## Imports and device setup

In [ ]:
import torch
import numpy as np
import rdv
import halort

rdv.device(0)  # activates GPU 0 (vulky/rdv's own device, independent of torch's CUDA device)
print("halort models available:", list(halort._MODEL_CLASSES))


## A synthetic cloud volume

`set_extinction` expects a `(D, H, W, 1)` density grid. We build a soft,
noisy blob here purely with numpy - swap this cell for a real volume
(e.g. from the upcoming Hugging Face dataset) once available.

In [ ]:
def make_synthetic_cloud(size=96, seed=0):
    rng = np.random.default_rng(seed)
    d, h, w = size, size, size
    zz, yy, xx = np.mgrid[0:d, 0:h, 0:w].astype(np.float32)
    center = np.array([d, h, w]) / 2
    zz, yy, xx = zz - center[0], yy - center[1], xx - center[2]
    radius = size * 0.32
    dist = np.sqrt(xx**2 + (yy * 1.3)**2 + zz**2)
    base = np.clip(1.0 - dist / radius, 0.0, 1.0) ** 1.5

    # a few octaves of smooth value noise, added on top for a "puffy" look
    noise = np.zeros((d, h, w), dtype=np.float32)
    amp = 1.0
    for octave_size in (8, 16, 32):
        coarse = rng.random((octave_size, octave_size, octave_size)).astype(np.float32)
        upsampled = torch.nn.functional.interpolate(
            torch.from_numpy(coarse)[None, None], size=(d, h, w), mode="trilinear", align_corners=False
        )[0, 0].numpy()
        noise += amp * (upsampled - 0.5)
        amp *= 0.5

    density = np.clip(base + 0.6 * noise * base, 0.0, None)
    return torch.from_numpy(density[..., None]).float()

cloud = make_synthetic_cloud().cuda()
print("cloud shape:", cloud.shape, "max density:", cloud.max().item())


## A synthetic environment + light

`set_environment` expects an `(H, W, 3)` HDR-like tensor (linear
radiance, no tone-mapping). We build a simple sky gradient here.

In [ ]:
def make_synthetic_sky(h=128, w=256):
    yy = np.linspace(1.0, 0.0, h, dtype=np.float32)[:, None]
    horizon = np.array([0.9, 0.85, 0.75], dtype=np.float32)
    zenith = np.array([0.25, 0.45, 0.85], dtype=np.float32)
    sky = horizon[None, None, :] * (1 - yy)[..., None] + zenith[None, None, :] * yy[..., None]
    sky = np.broadcast_to(sky, (h, w, 3)).copy()
    return torch.from_numpy(sky).float()

environment = make_synthetic_sky().cuda()
light_direction = (0.3, 0.8, 0.4)
light_intensity = (3.0, 2.9, 2.7)


## Build and configure a model

Same scene-setup API for all three model types.

In [ ]:
def setup_scene(net):
    net.set_extinction(cloud, density_scale=300.0)
    net.set_scattering_albedo((0.99, 0.99, 0.99))
    net.set_anisotropy(0.3)
    net.set_environment(environment)
    net.set_camera(0, origin=(0.0, 0.5, 2.6), target=(0.0, 0.0, 0.0), up=(0.0, 1.0, 0.0))
    net.set_light(light_direction=light_direction, light_intensity=light_intensity)
    return net


## HaloRT

Needs the paired `halo_dir.pt` / `halo_env.pt` checkpoint files from
`pretrained/`.

In [ ]:
halort_net = halort.create_scatnet("halort", width=512, height=512).cuda()
halort_net.load_pretrained("pretrained/halo_dir.pt", "pretrained/halo_env.pt")
setup_scene(halort_net)

with torch.no_grad():
    halort_image = halort_net.reconstruct_radiance_fused()  # (1, H, W, 3), background-composited

print(halort_image.shape)


## Denoiser / DenoiserX

In [ ]:
denoiser_net = halort.create_scatnet("denoiser", width=512, height=512).cuda()
denoiser_net.load_pretrained("pretrained/denoiser.pt")
setup_scene(denoiser_net)

denoiserx_net = halort.create_scatnet("denoiserx", width=512, height=512).cuda()
denoiserx_net.load_pretrained("pretrained/denoiserx.pt")
setup_scene(denoiserx_net)

with torch.no_grad():
    denoiser_image = denoiser_net.capture_reconstructed_radiance()
    denoiserx_image = denoiserx_net.capture_reconstructed_radiance()

print(denoiser_image.shape, denoiserx_image.shape)


## Display side by side

In [ ]:
import matplotlib.pyplot as plt

def tonemap(img):
    a = img[0].detach().float().cpu().numpy()
    return np.clip(np.clip(a, 0.0, None) ** (1.0 / 2.2), 0.0, 1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, name, img in zip(axes, ["HaloRT", "Denoiser", "DenoiserX"], [halort_image, denoiser_image, denoiserx_image]):
    ax.imshow(np.flipud(tonemap(img)))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()


## Next steps

- Swap `make_synthetic_cloud()` for a real volume once the Hugging Face
  cloud dataset reference is available (see the main README).
- Try changing `density_scale`, `scattering_albedo`, `g` (anisotropy),
  `light_direction`/`light_intensity`, and the camera pose in
  `setup_scene()`.
- See `02_live_rendering.ipynb` for an interactive / animated example.